# Run the GTA tracking pipeline on Colab

Two-stage pipeline: **DeepEIoU tracking → GtaLink refinement**. Reads input videos from Google
Drive and writes every output back to Drive.

- **Input videos:** `Colab Notebook/football-analysis-project/input-videos/`
- **Outputs:** `Colab Notebook/football-analysis-project/output/gta-track/<video>/`

See `COLAB_GUIDE.md` (running on Colab) and `USER_GUIDE.md` (what each stage/flag does) in the repo.

**Before you run:**
1. Use a GPU runtime: *Runtime → Change runtime type → Hardware accelerator → GPU*.
2. The code is on the `pipeline-refactor` branch — make sure it's pushed to GitHub so the clone
   cell works (`git push -u origin pipeline-refactor`).
3. Checkpoints download automatically in step 4 (public Drive folder from the Deep-EIoU readme).

## 0. Verify the GPU runtime

In [ ]:
!nvidia-smi

## 1. Mount Drive and define paths

Edit `DRIVE_BASE` if your Drive folder is actually **`Colab Notebooks`** (plural) or named
differently, and set `VIDEO` to a file that exists under `input-videos/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# --- EDIT THIS if your Drive folder is "Colab Notebooks" (plural) ---
DRIVE_BASE = "/content/drive/MyDrive/Colab Notebook/football-analysis-project"

INPUT_DIR  = f"{DRIVE_BASE}/input-videos"
OUTPUT_DIR = f"{DRIVE_BASE}/output/gta-track"
CKPT_DIR   = f"{DRIVE_BASE}/checkpoints"
REPO       = "/content/football-analysis-2"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pick the clip to process (must exist under INPUT_DIR):
VIDEO = f"{INPUT_DIR}/M59-5min-1.mp4"     # <-- change to your file
STEM  = Path(VIDEO).stem

# Export to the shell environment so the `!` cells below can use them:
os.environ.update(INPUT_DIR=INPUT_DIR, OUTPUT_DIR=OUTPUT_DIR,
                  CKPT_DIR=CKPT_DIR, REPO=REPO, VIDEO=VIDEO, STEM=STEM)

print("Input videos available:")
!ls -la "$INPUT_DIR"

## 2. Clone the code

Keeps the repo on Colab's fast local disk (`/content`); only outputs go to Drive. (If you'd rather
not push to GitHub, copy/unzip the project into `/content/football-analysis-2` instead.)

In [ ]:
!git clone -b pipeline-refactor https://github.com/luna4tech/football-analysis-2.git "$REPO"
%cd $REPO
!git log --oneline -3

## 3. Install dependencies

One shared environment serves **both** stages. Colab already has `torch`/`torchvision`; install
GtaLink's `requirements.txt` (covers Stage 2's refine libs **and** the import-time deps of the
bundled torchreid that Stage 1's ReID uses), then add `cython_bbox` (the tracker's IoU) + `tqdm`,
which it doesn't list.

You do **not** need the READMEs' build steps: Deep-EIoU's `setup.py` only compiles `yolox._C`
(COCO-eval only), `reid/setup.py` only builds an optional Cython metric with a fallback, and
`pip install torchreid` is irrelevant — `reid` is the **in-repo** `Deep-EIoU/Deep-EIoU/reid/`
folder, used in place via Stage 1's working directory, never pip-installed.

In [ ]:
%cd $REPO
!pip install -q -r gta-link/requirements.txt
!pip install -q cython_bbox tqdm

## 4. Fetch the checkpoints (once, to Drive) and link them

Downloads `best_ckpt.pth.tar` (detector) and `sports_model.pth.tar-60` (ReID) from the public Drive
folder the first time, caches them in your Drive `checkpoints/` folder, and symlinks them into the
directory Stage 1 expects (`Deep-EIoU/Deep-EIoU/checkpoints/`).

In [ ]:
import glob, os

os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_FOLDER_URL = "https://drive.google.com/drive/folders/1wItcb0yeGaxOS08_G9yRWBTnpVf0vZ2w"

# Download only if not already in Drive (large files — avoid re-downloading each session).
have = {os.path.basename(p) for p in glob.glob(f"{CKPT_DIR}/**/*", recursive=True)}
if not {"best_ckpt.pth.tar", "sports_model.pth.tar-60"} <= have:
    !pip install -q gdown
    !gdown --folder "{CKPT_FOLDER_URL}" -O "$CKPT_DIR"

# Link each checkpoint into Deep-EIoU/Deep-EIoU/checkpoints/ (robust to any sub-folder gdown made).
dst_dir = f"{REPO}/Deep-EIoU/Deep-EIoU/checkpoints"
os.makedirs(dst_dir, exist_ok=True)
for name in ("best_ckpt.pth.tar", "sports_model.pth.tar-60"):
    matches = glob.glob(f"{CKPT_DIR}/**/{name}", recursive=True)
    assert matches, f"{name} not found under {CKPT_DIR} - check the gdown download"
    dst = f"{dst_dir}/{name}"
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(matches[0], dst)
    print("linked", matches[0], "->", dst)

!ls -la "{dst_dir}"

## 5. Run the full pipeline → outputs to Drive

Runs Stage 1 (tracking) then Stage 2 (refine), writing everything under `OUTPUT_DIR/<STEM>/`. The
orchestrator switches each stage's working directory internally and prints a profiling summary.

In [ ]:
%cd $REPO
!python -m pipeline run \
    --video "$VIDEO" \
    --artifacts-dir "$OUTPUT_DIR" \
    --device gpu

**Optional — faster perception (batched detect+ReID + prefetch decode).** Lower `--batch-size`
if you hit GPU out-of-memory.

> **Cache note:** re-running the same video is instant (cached). Changing a *parameter* does NOT
> bust the cache — add `--force-all`, `--force-stage1`, or `--force-stage2`. E.g. retune refine
> only: `... --eps 0.5 --merge_dist_thres 0.3 --force-stage2`.

In [ ]:
!python -m pipeline run \
    --video "$VIDEO" \
    --artifacts-dir "$OUTPUT_DIR" \
    --device gpu \
    --parallel --batch-size 8

## 6. (Optional) Render an annotated video to Drive

The pipeline outputs MOT `.txt`, not a video. Rebuild a watchable overlay from the refined result
(CPU only). Frames are 0-based, so do **not** pass `--one_indexed`.

In [ ]:
%cd $REPO/Deep-EIoU/Deep-EIoU
!python tools/render_from_txt.py \
    --path "$VIDEO" \
    --txt  "$OUTPUT_DIR/$STEM/02_refine/refined.txt" \
    --save_path "$OUTPUT_DIR/$STEM/${STEM}_refined.mp4"
%cd $REPO

## 7. Verify the outputs

Track-count before vs after refinement (refined should have fewer / cleaner IDs), then the profiling
summary (per-stage time / GPU mem / CPU mem / IO counts).

In [ ]:
%cd $REPO/Deep-EIoU/Deep-EIoU
!python tools/count_tracks.py "$OUTPUT_DIR/$STEM/01_track/tracks.txt"
!python tools/count_tracks.py "$OUTPUT_DIR/$STEM/02_refine/refined.txt"
%cd $REPO
!cat "$OUTPUT_DIR/$STEM/profiles/summary.md"

**Optional — confirm `--parallel` matches sequential.** Run each mode into a *separate* artifacts
dir, then diff the two `tracks.txt` with the tolerant comparator (parity is exact only up to
floating-point noise).

In [ ]:
!python -m pipeline run --video "$VIDEO" --artifacts-dir /content/_seq --device gpu --force-stage1
!python -m pipeline run --video "$VIDEO" --artifacts-dir /content/_par --device gpu --parallel --batch-size 8 --force-stage1
!python -m pipeline compare \
    --a "/content/_seq/$STEM/01_track/tracks.txt" \
    --b "/content/_par/$STEM/01_track/tracks.txt" \
    --tol 1.0

## Output layout on Drive

```
output/gta-track/<STEM>/
  01_track/tracks.txt        # raw DeepEIoU tracking (MOT, 0-based frames)
  01_track/tracklets.pkl     # per-id tracklets WITH reused ReID features (large)
  02_refine/refined.txt      # GtaLink split+connect output (final result)
  profiles/01_track.json     # Stage 1 profile (time / GPU / CPU / counts)
  profiles/02_refine.json    # Stage 2 profile
  profiles/summary.{json,md}
  <STEM>_refined.mp4         # only if you ran the optional render step
```

**Tips:** OOM in `--parallel` → lower `--batch-size` (4/2) or drop `--parallel`. Slow/flaky Drive
writes on long videos → run with `--artifacts-dir /content/work` then `!cp -r /content/work/$STEM
"$OUTPUT_DIR/"`. Keep the `"$VAR"` quotes (the Drive path has a space).